## 1. Устанавливаю спарк на 1 же окружение

In [ ]:
# проверяю какое окружение стоит
import sys
print("Jupyter python:", sys.executable)

Jupyter python: /Users/user/Full_ML_Course/.venv/bin/python


In [2]:
# привязываю спарк к этому же окружению
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("PYSPARK_PYTHON:", os.environ["PYSPARK_PYTHON"])
print("PYSPARK_DRIVER_PYTHON:", os.environ["PYSPARK_DRIVER_PYTHON"])

PYSPARK_PYTHON: /Users/user/Full_ML_Course/.venv/bin/python
PYSPARK_DRIVER_PYTHON: /Users/user/Full_ML_Course/.venv/bin/python


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("check-python").getOrCreate()

print("spark.pyspark.python =", spark.sparkContext.getConf().get("spark.pyspark.python"))
print("spark.pyspark.driver.python =", spark.sparkContext.getConf().get("spark.pyspark.driver.python"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/04 15:58:21 WARN Utils: Your hostname, MacBook--Azich.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.2 instead (on interface en0)
26/02/04 15:58:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/04 15:58:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


spark.pyspark.python = None
spark.pyspark.driver.python = None


In [4]:
r = spark.sparkContext.parallelize([1], 1).map(lambda _: __import__("sys").executable).collect()
print("Executor python:", r[0])

Executor python: /Users/user/Full_ML_Course/.venv/bin/python


In [5]:
df = spark.createDataFrame(
    [("Betty_White", 288886), ("Main_Page", 139564)],
    ["article_title", "view_count"]
)
df.show()

+-------------+----------+
|article_title|view_count|
+-------------+----------+
|  Betty_White|    288886|
|    Main_Page|    139564|
+-------------+----------+



In [6]:
spark.stop()

## 2. Работа на спарке

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate() 

### Spark SQL

In [7]:
spark = SparkSession.builder\
    .config('spark.app.name', 'learning_spark_sql')\
    .getOrCreate()
    
print(spark.sparkContext) 
# <SparkContext master=local[*] appName=learning_spark_sql>

<SparkContext master=local[*] appName=learning_spark_sql>


In [ ]:
# Create an RDD from a list
hrly_views_rdd  = spark.sparkContext.parallelize([
    ["Betty_White" , 288886],
    ["Main_Page", 139564],
    ["New_Year's_Day", 7892],
    ["ABBA", 8154]
])

# Convert RDD to DataFrame
hrly_views_df = hrly_views_rdd\
    .toDF(["article_title", "view_count"])
    
hrly_views_df.show(4, truncate=False)

+--------------+----------+
|article_title |view_count|
+--------------+----------+
|Betty_White   |288886    |
|Main_Page     |139564    |
|New_Year's_Day|7892      |
|ABBA          |8154      |
+--------------+----------+



In [11]:
# Доступ к RDD, лежащему в основе DataFrame
hrly_views_df_rdd = hrly_views_df.rdd

# Проверьте тип объекта
print(type(hrly_views_df_rdd)) 
# <класс 'pyspark.rdd.RDD'>

<class 'pyspark.core.rdd.RDD'>


In [ ]:
print(type(spark.read)) 
# <class 'pyspark.sql.readwriter.DataFrameReader'>

# Read CSV to DataFrame
hrly_views_df = spark.read\
.option('header', True) \
.option('delimiter', ' ') \
.option('inferSchema', True)  \
.csv('views_2022_01_01_000000.csv')

# Display first 5 rows of DataFrame
hrly_views_df.show(5, truncate=False)

<class 'pyspark.sql.readwriter.DataFrameReader'>


26/02/04 16:24:09 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: views_2022_01_01_000000.csv.
java.io.FileNotFoundException: File views_2022_01_01_000000.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/Users/user/Full_ML_Course/27-PySpark/views_2022_01_01_000000.csv. SQLSTATE: 42K03

26/02/04 17:08:10 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 932390 ms exceeds timeout 120000 ms
26/02/04 17:08:10 WARN SparkContext: Killing executors is not supported by current scheduler.
26/02/04 17:08:15 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1363)
	at o

In [4]:
spark.stop()